In [26]:
"""
Vorbereitung
step 1) erstmal Dateien extrahieren
step 2) manuell in VS txt-format in csv umwandeln (einfach mit rechtsklick + rename)
"""

'\nVorbereitung\nstep 1) erstmal Dateien extrahieren\nstep 2) manuell in VS txt-format in csv umwandeln (einfach mit rechtsklick + rename)\n'

In [27]:

import pandas as pd
import numpy as np


# name = 'produkt_zehn_min_sd_20000101_20080526_00222.csv' -> war erster Versuch

name = 'produkt_zehn_min_sd_20200101_20241231_00282.csv'

# CSV-Datei laden
df = pd.read_csv(name,sep=';')
df



,STATIONS_ID,MESS_DATUM,QN,DS_10,GS_10,SD_10,LS_10,eor
0,282,202001010000,3,0.0,0.0,0.0,-999,eor
1,282,202001010010,3,0.0,0.0,0.0,-999,eor
2,282,202001010020,3,0.0,0.0,0.0,-999,eor
3,282,202001010030,3,0.0,0.0,0.0,-999,eor
4,282,202001010040,3,0.0,0.0,0.0,-999,eor
...,...,...,...,...,...,...,...,...
263083,282,202412312310,3,0.0,0.0,0.0,-999,eor
263084,282,202412312320,3,0.0,0.0,0.0,-999,eor
263085,282,202412312330,3,0.0,0.0,0.0,-999,eor
263086,282,202412312340,3,0.0,0.0,0.0,-999,eor


In [28]:
df_neu = df

df_neu['MESS_DATUM'] = pd.to_datetime(df['MESS_DATUM'], format='%Y%m%d%H%M')
df_neu


,STATIONS_ID,MESS_DATUM,QN,DS_10,GS_10,SD_10,LS_10,eor
0,282,2020-01-01 00:00:00,3,0.0,0.0,0.0,-999,eor
1,282,2020-01-01 00:10:00,3,0.0,0.0,0.0,-999,eor
2,282,2020-01-01 00:20:00,3,0.0,0.0,0.0,-999,eor
3,282,2020-01-01 00:30:00,3,0.0,0.0,0.0,-999,eor
4,282,2020-01-01 00:40:00,3,0.0,0.0,0.0,-999,eor
...,...,...,...,...,...,...,...,...
263083,282,2024-12-31 23:10:00,3,0.0,0.0,0.0,-999,eor
263084,282,2024-12-31 23:20:00,3,0.0,0.0,0.0,-999,eor
263085,282,2024-12-31 23:30:00,3,0.0,0.0,0.0,-999,eor
263086,282,2024-12-31 23:40:00,3,0.0,0.0,0.0,-999,eor


In [29]:

#start = df_neu[df_neu['MESS_DATUM'] == '2004-01-01 00:00'].index --> ist schlecht lieber gleich mit pd.Timestemp Attribut arbeiten
#ende = df_neu[df_neu['MESS_DATUM'] == '2004-12-31 23:50'].index
#Versuche:
#df_neu['MESS_DATUM'] = df_neu[(df_neu['MESS_DATUM'] >= '2004-01-01 00:00') & (df_neu['MESS_DATUM'] < '2004-12-31 23:50')]
#df_neu = df_neu[(df_neu['MESS_DATUM'] >= start[0]) & (df_neu['MESS_DATUM'] <= ende[0])]
#df_neu['MESS_DATUM'][start:ende]


In [30]:
df_neu['MESS_DATUM'] = pd.to_datetime(df_neu['MESS_DATUM'], format='%Y-%m-%d %H:%M')

# eigentlich ist 2004 richtig, hier wird 2024 verwendet für Proof of concept
# start = pd.Timestamp('2004-01-01 00:00')
# ende = pd.Timestamp('2004-12-31 23:50')
start = pd.Timestamp('2024-01-01 00:00')
ende = pd.Timestamp('2024-12-31 23:50')


# Filtere den DataFrame im gewünschten Zeitraum
df_neu = df_neu[(df_neu['MESS_DATUM'] >= start) & (df_neu['MESS_DATUM'] <= ende)]
df_neu


,STATIONS_ID,MESS_DATUM,QN,DS_10,GS_10,SD_10,LS_10,eor
210384,282,2024-01-01 00:00:00,3,0.0,0.0,0.0,-999,eor
210385,282,2024-01-01 00:10:00,3,0.0,0.0,0.0,-999,eor
210386,282,2024-01-01 00:20:00,3,0.0,0.0,0.0,-999,eor
210387,282,2024-01-01 00:30:00,3,0.0,0.0,0.0,-999,eor
210388,282,2024-01-01 00:40:00,3,0.0,0.0,0.0,-999,eor
...,...,...,...,...,...,...,...,...
263083,282,2024-12-31 23:10:00,3,0.0,0.0,0.0,-999,eor
263084,282,2024-12-31 23:20:00,3,0.0,0.0,0.0,-999,eor
263085,282,2024-12-31 23:30:00,3,0.0,0.0,0.0,-999,eor
263086,282,2024-12-31 23:40:00,3,0.0,0.0,0.0,-999,eor


In [31]:
#Transition von J/cm^2 [DWD] zu W/m^2 [PVSyst]
# 
# 1W = 1J/s 
# 1m^2 = 100^2 cm^2
# alle Werte sind in 15 Minuten Schritten
# --> 1 J/cm^2 = 1/(60[s->min]*15[timestep]*100^2[Fläche]) W/m^2

energy_to_power_15min = 1/(60*15*100^2) # J/cm^2 to W/m^2 [15min]
# energy_to_power_1h    = 1/(60^2 *100^2) # J/cm^2 to W/m^2 [1h] # später addiere ich ja werte und teile sie durch 4 --> braucht nur einen Transition Faktor


In [32]:
#df_neu = df_neu.drop(columns=['STATIONS_ID'])
#df_neu = df_neu.drop(columns=['QN'])

df_neu_relevant = df_neu[['MESS_DATUM','GS_10']]
df_neu_relevant 

df_neu_relevant['Year'] = df_neu['MESS_DATUM'].dt.year
df_neu_relevant['Month'] = df_neu['MESS_DATUM'].dt.month
df_neu_relevant['Day'] = df_neu['MESS_DATUM'].dt.day
df_neu_relevant['Hour'] = df_neu['MESS_DATUM'].dt.hour
df_neu_relevant['Minute'] = df_neu['MESS_DATUM'].dt.minute


#bisschen unschön, aber dann passt es: GHI ans Ende
# hier sollte es noch GS_10 heißen!
# df_neu_relevant['GHI'] = df_neu['GS_10'] # nochmal gegenchecken 


df_neu_relevant.drop(columns=['MESS_DATUM'], inplace=True)
#df_neu_relevant.drop(columns=['GS_10'], inplace=True)

df_neu_relevant

#   -->  Ziel Format: ;;;;;W/m2;W/m2;deg.C

C:\Users\CasparGross\AppData\Local\Temp\ipykernel_1716\2786760943.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_neu_relevant['Year'] = df_neu['MESS_DATUM'].dt.year
C:\Users\CasparGross\AppData\Local\Temp\ipykernel_1716\2786760943.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_neu_relevant['Month'] = df_neu['MESS_DATUM'].dt.month
C:\Users\CasparGross\AppData\Local\Temp\ipykernel_1716\2786760943.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.


,GS_10,Year,Month,Day,Hour,Minute
210384,0.0,2024,1,1,0,0
210385,0.0,2024,1,1,0,10
210386,0.0,2024,1,1,0,20
210387,0.0,2024,1,1,0,30
210388,0.0,2024,1,1,0,40
...,...,...,...,...,...,...
263083,0.0,2024,12,31,23,10
263084,0.0,2024,12,31,23,20
263085,0.0,2024,12,31,23,30
263086,0.0,2024,12,31,23,40


In [ ]:
#trimestamp-shift:
# Werte des DWD sind "Summe der Globalstrahlung der vorangegangnen 10 Minuten"
# Werte in PV Syst haben Zeitstempel zu Anfang des Intervalls
# Ziel ist also Verschieben des Zeitstempels um 10 Minuten zurück
# außerdem sollten Werte auf 15-Minuten Basis berechnet werden.
# da es sich bei Werten von DWD um Energiemenge handelt ist die sinnvollste Interpolation für 30 Minuten das 2. Zeitintervall jeweils zu halbieren und beiden anderen Zeitintervallen hinzu zu addieren.
# Der hierbei entstehende Fehler ist dann, dass die für einen Wert tendentiell überschätzt werden und den anderen Wert tendentiell unterschätzt werden, aber in Summe ist auf jeden Fall die erzeugte Energie korrekt.

def aus_10_min_mach_15(df): 
    # hier versuche ich mal die neuen dfs mit indicees zu berechnen und nicht wie sonst immer mit einer for Schleife, weil das viel schneller und sinnvoller ist.
    # dafür musste ich allerdings immer den Index neu setzen mit .reset_index(drop=True), da pd sonst keine Matrixoperationen durchführen konnte 
    indicees_15 = [i for i in range(len(df)-1) if i % 3 == 0 or i % 3 == 2]  # ich suche für 30 Minuten immer Minute 0 und 20 aus, damit sie die neuen 15 Minuten Werte werden
    df_15 = df.iloc[indicees_15].reset_index(drop=True) # jetzt df hiermit erzeugen
    indicees_15_aufteilen = [i for i in range(len(df)-1) if i % 3 == 1]  # jetzt wird für  30 Minuten immer Minute 20 ausgesucht 
    df_15_aufteilen = df['GS_10'].iloc[indicees_15_aufteilen].repeat(2).reset_index(drop=True)/2   # diese muss dann noch durch 2 geteilt werden, um die Hälfte dem ersten Wert und die andere Hälfte dem zweiten Wert zuzuordnen 
    df_15['GS_10'] =  df_15['GS_10'] + df_15_aufteilen
    df_15['GHI'] = df_15['GS_10'] * energy_to_power_15min
    df_15.drop(columns=['GS_10'], inplace=True)
    
    # hier Transition Faktor einbeziehen
    
    #indicees_minutes = [i for i in range(len(df_15)-1) if i % 3 == 2] # hier wähle ich wieder jeden zweiten Wert aus, um die Minuten anzupassen
    #hier musste ich leider die for-Schleife wählen, weil es irgendwarum nicht anders funktioniert hat...
    for i in range(int(len(df_15)/2)):
        j= i*2 +1
        df_15['Minute'].iloc[j] = df_15['Minute'].iloc[j] -5

    #df_15['Minute'].iloc[indicees_minutes] = df_15['Minute'] - minus_fünf   # Reste von früheren Versuchen. Wenn der Code mal automatisiert irgendwo eingebaut werden sollte, dann müsste man hier noch die for-Schleife erstezen.
    #df_1h = df_15.iloc[::2].reset_index(drop=True)  # hier wähle ich dann immer jeden zweiten Wert aus, um die Stundenwerte zu bekommen
    
    return df_15 
    


df_15   = aus_10_min_mach_15(df_neu_relevant)
df_15


C:\Users\CasparGross\AppData\Local\Temp\ipykernel_1716\1386646058.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_15['Minute'].iloc[j] = df_15['Minute'].iloc[j] -5
C:\Users\CasparGross\AppData\Local\Temp\ipykernel_1716\1386646058.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_15['Minute'].iloc[j] = df_15['Minute'].iloc[j] -5
C:\Users\CasparGross\AppData\Local\Temp\ipykernel_1716\1386646058.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning

,Year,Month,Day,Hour,Minute,GHI
0,2024,1,1,0,0,0.0
1,2024,1,1,0,15,0.0
2,2024,1,1,0,30,0.0
3,2024,1,1,0,45,0.0
4,2024,1,1,1,0,0.0
...,...,...,...,...,...,...
35130,2024,12,31,22,30,0.0
35131,2024,12,31,22,45,0.0
35132,2024,12,31,23,0,0.0
35133,2024,12,31,23,15,0.0


In [34]:

# # indicees_fuer_1h = np.zeros([3,round(len(df_15)/4)])  # nicht wirklich resillient, da indicee-Längen manchmal unterschiedlich sind.


# hier einmal indicees erstellen, damit keine for-Schleife genutzt werden muss. --> Logik ist einfach: aus 15_min_Werten müssen 1h Werte entstehen --> wir wollen immer einstündig versetzte Einträge 
indicees_fuer_1h_1= np.array([i for i in range(len(df_15)-1) if i % 4 == 0])
indicees_fuer_1h_2= np.array([i for i in range(len(df_15)-1) if i % 4 == 1])
indicees_fuer_1h_3= np.array([i for i in range(len(df_15)-1) if i % 4 == 2])
indicees_fuer_1h_4= np.array([i for i in range(len(df_15)-1) if i % 4 == 3])


# hier einmal den df für 1h erstellen, indem wir die Einträge der vollen Stunde übernehmen.
df_1h = df_15.iloc[indicees_fuer_1h_1].reset_index(drop=True)  
# dann als nächstes die Globalstrahlung der 15 Minuten Werte aufaddieren und durch 4 teilen (ist ja ne Leistung) --> hierfür haben wir ja schon die Indicees und .reset_index ist wichtig, weil pd die df nicht matchen kann 
df_1h['GHI'] = (df_15['GHI'].iloc[indicees_fuer_1h_1].reset_index(drop=True) + df_15['GHI'].iloc[indicees_fuer_1h_2].reset_index(drop=True) + df_15['GHI'].iloc[indicees_fuer_1h_3].reset_index(drop=True) + df_15['GHI'].iloc[indicees_fuer_1h_4].reset_index(drop=True)) /4


# Muss unbedingt noch geändert werden, nur für Proof of concept

df_1h['Tamb'] = 15

df_1h
# hier sieht man allerdings schon die Ungenauigkeit des Stundensystems: während im 15 Minuten System der Sonnenaufgang zummindest noch auf 7:30 Uhr angesetzt ist, ist er im Stundensystem schon auf 7:00 Uhr gesetzt.


,Year,Month,Day,Hour,Minute,GHI,Tamb
0,2024,1,1,0,0,0.0,15
1,2024,1,1,1,0,0.0,15
2,2024,1,1,2,0,0.0,15
3,2024,1,1,3,0,0.0,15
4,2024,1,1,4,0,0.0,15
...,...,...,...,...,...,...,...
8779,2024,12,31,19,0,0.0,15
8780,2024,12,31,20,0,0.0,15
8781,2024,12,31,21,0,0.0,15
8782,2024,12,31,22,0,0.0,15


In [35]:
# W/m^2 ist auf alle Fälle nicht J/cm^2 --> es muss also noch Faktor berücksichtigt werden
new_row = ['', '', '', '', '', 'W/m2', 'deg.C']   # 'deg.C' und gesamte Spalte für Temp fehlt noch

# einheiten einfügen bei CSV -> entkommentieren der beiden Zeilen unten
# df_15 = pd.concat([pd.DataFrame([new_row], columns=df_15.columns), df_15], ignore_index=True) # nur einmal einfügen
# df_1h = pd.concat([pd.DataFrame([new_row], columns=df_1h.columns), df_1h], ignore_index=True) # nur einmal einfügen
df_1h


,Year,Month,Day,Hour,Minute,GHI,Tamb
0,2024,1,1,0,0,0.0,15
1,2024,1,1,1,0,0.0,15
2,2024,1,1,2,0,0.0,15
3,2024,1,1,3,0,0.0,15
4,2024,1,1,4,0,0.0,15
...,...,...,...,...,...,...,...
8779,2024,12,31,19,0,0.0,15
8780,2024,12,31,20,0,0.0,15
8781,2024,12,31,21,0,0.0,15
8782,2024,12,31,22,0,0.0,15


In [36]:
#achtung! name muss noch im cache sein!

name_new = name.replace('.csv', '_2024_15min_for_pv_syst.csv')

# nur ausführen wenn Datei ready ist, sonst wird bestehende Datei überschrieben 
df_1h.to_csv(name_new, sep=';', index=False, header=['Year','Month','Day','Hour','Minute','GHI','Tamb'])



#   diese Header am besten noch integrieren, geht aber auch händisch

#Site; Schleswig
#Country; germany
#Data Source; DWD_station_4466 
#Time step; 1h
#Year; 2024
#Latitude; 54.5275
#Longitude; 9.5487
#Altitude; 42.70
#Time Zone; 1  

# --> muss aber individuell je nach Metadaten angepasst werden (könnte man aber in Zukunft automatisieren)

# Zielformat
# Year;Month;Day;Hour;Minute;GHI;Tamb;
# ;;;;;W/m2;deg.C;

In [37]:
"""
ToDo:

# location checken
# Windgeschwindigkeit u Temp ergänzen --> müsste aus Metadaten hervorgehen, wo die zu finden sind.

# Simulieren mit Lima

# Wetterstationen filtern später

UTF-Zeitformat anpassen --> bisher ist noch nicht der Switch zwischen UTF + 1 und local Time integriert
wenn ich das aber richtig aus dr Doku lese, dann reicht es am Anfang #Time Reference; 1  zu ergänzen --> das müsste für PV-Syst dann klar stellen, dass es selbst den Shift von UTF 1 zu local time macht

https://www.pvsyst.com/help/meteo-database/notes-on-meteo/meteonote7-time-shift-in-meteo-files.html#time-shift-correction-when-importing-data 
If there is a time shift varying along the year:

With the general ASCII importing tool, consider using the DST correction (option "Summer Time > Apply").
Otherwise there is no practical way, except modifying the dates directly in the source file.
If the file is written in UT (Universal Time):

With the general ASCII importing tool, check the box "Universal time".
With PVsyst standard format, you can add a line #Time Reference with the value UT

"""

'\nToDo:\n\n# location checken\n# Windgeschwindigkeit u Temp ergänzen --> müsste aus Metadaten hervorgehen, wo die zu finden sind.\n\n# Simulieren mit Lima\n\n# Wetterstationen filtern später\n\nUTF-Zeitformat anpassen --> bisher ist noch nicht der Switch zwischen UTF + 1 und local Time integriert\nwenn ich das aber richtig aus dr Doku lese, dann reicht es am Anfang #Time Reference; 1  zu ergänzen --> das müsste für PV-Syst dann klar stellen, dass es selbst den Shift von UTF 1 zu local time macht\n\nhttps://www.pvsyst.com/help/meteo-database/notes-on-meteo/meteonote7-time-shift-in-meteo-files.html#time-shift-correction-when-importing-data \nIf there is a time shift varying along the year:\n\nWith the general ASCII importing tool, consider using the DST correction (option "Summer Time > Apply").\nOtherwise there is no practical way, except modifying the dates directly in the source file.\nIf the file is written in UT (Universal Time):\n\nWith the general ASCII importing tool, check the 